In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)


tensor([[-0.1507, -0.2135,  0.0078, -0.0088, -0.2327,  0.0164,  0.0049,  0.0158,
         -0.1030,  0.0650],
        [-0.2849, -0.2723, -0.1283,  0.0111, -0.1742, -0.0177,  0.1929,  0.0593,
         -0.1618,  0.1314]], grad_fn=<AddmmBackward0>)

自定义块

In [2]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        # 注意，这里我们使用ReLU的函数版本，其在nn.functional模块中定义。
        return self.out(F.relu(self.hidden(X)))


实例化使用

In [3]:
net = MLP()
net(X)


tensor([[ 0.1432,  0.1421,  0.0866,  0.0481, -0.2320,  0.1805,  0.0186,  0.0373,
          0.2040,  0.1328],
        [ 0.1275,  0.1165,  0.0715,  0.2159, -0.1709,  0.1258,  0.0139, -0.1585,
          0.2375,  0.0718]], grad_fn=<AddmmBackward0>)

顺序块

In [5]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)


tensor([[-0.0633, -0.1322, -0.0711,  0.2580,  0.0666, -0.1411, -0.2617, -0.0146,
          0.0652,  0.0575],
        [-0.1236, -0.2576, -0.1473,  0.2334,  0.1315, -0.0329, -0.1003, -0.2245,
          0.0075, -0.0599]], grad_fn=<AddmmBackward0>)

在正向传播函数中执行代码

In [12]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()
net = FixedHiddenMLP()
net(X)


tensor(0.0894, grad_fn=<SumBackward0>)

混合搭配各种组合块的方法

In [13]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)


tensor(0.0194, grad_fn=<SumBackward0>)